<a href="https://colab.research.google.com/github/kunphat510214-netizen/project6/blob/Step-7/step_07_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ขั้นที่ 7: วิเคราะห์ข้อมูลด้วย pandas และ SQL

ก่อนรัน ให้วาง `coffee_orders.csv` และ `coffee_shop.db` ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊ก หรืออัปโหลดเข้า Colab ก่อน


In [ ]:
"""ขั้นที่ 7: วิเคราะห์ข้อมูลร้านกาแฟด้วย pandas และ SQL."""

from pathlib import Path
import sqlite3

import pandas as pd


CSV_PATH = Path("coffee_orders.csv")
DATABASE_PATH = Path("coffee_shop.db")


def show(title, dataframe):
    """แสดง DataFrame ให้อ่านได้ทั้งใน terminal และ GitHub Codespaces."""
    print(f"\n{title}")
    print(dataframe.to_string(index=False))


if not CSV_PATH.exists():
    raise FileNotFoundError("ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน")
if not DATABASE_PATH.exists():
    raise FileNotFoundError("ไม่พบ coffee_shop.db กรุณารันขั้นที่ 5-6 ก่อน")


# 7.1 โหลด CSV และสำรวจข้อมูลเบื้องต้น
coffee_df = pd.read_csv(CSV_PATH)
coffee_df.info()
show("สถิติเบื้องต้น", coffee_df.describe(include="all").reset_index())

In [ ]:
# 7.2 วิเคราะห์ด้วย pandas: groupby + agg + sort_values + Top 5
menu_summary = (
    coffee_df.groupby("menu_name")
    .agg(
        total_orders=("order_id", "count"),
        total_revenue=("price", "sum"),
        avg_price=("price", "mean"),
        avg_wait_minutes=("wait_minutes", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
show("Top 5 เมนูที่สร้างรายได้สูงสุด", menu_summary.head(5))

In [ ]:
# 7.3 วิเคราะห์ด้วย SQL
queries = {
    "ออเดอร์ราคาสูง": """
        SELECT order_id, queue_no, menu_name, price
        FROM orders
        WHERE price >= 75
        ORDER BY price DESC
        LIMIT 5
    """,
    "QR Code และกลับบ้าน": """
        SELECT order_id, menu_name, receive_type, payment_method, price
        FROM orders
        WHERE receive_type = 'กลับบ้าน'
          AND payment_method = 'QR Code'
        ORDER BY order_id
        LIMIT 5
    """,
    "คิวรอนาน": """
        SELECT order_id, menu_name, wait_minutes
        FROM orders
        WHERE wait_minutes >= 15
        ORDER BY wait_minutes DESC, order_id
        LIMIT 5
    """,
    "รายได้ตามช่องทางชำระ": """
        SELECT payment_method,
               COUNT(*) AS total_orders,
               ROUND(SUM(price), 2) AS total_revenue,
               ROUND(AVG(price), 2) AS avg_price
        FROM orders
        GROUP BY payment_method
        ORDER BY total_revenue DESC
    """,
    "JOIN สมาชิกกับออเดอร์": """
        SELECT o.order_id, o.queue_no, m.member_name, o.menu_name, o.price
        FROM orders AS o
        JOIN members AS m ON o.member_id = m.member_id
        WHERE o.price >= 60
        ORDER BY o.price DESC
        LIMIT 5
    """,
}

with sqlite3.connect(DATABASE_PATH) as connection:
    for title, query in queries.items():
        show(title, pd.read_sql_query(query, connection))